# Compute: Robustness Benchmark

Runs benchmark for Figure 2. Output: robustness_benchmark.json

Run this first, then fig2_robustness.ipynb for plotting.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import json, os, re, subprocess, time, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DATA_DIR = Path('/benchmark/data/generated')
RESULTS_DIR = Path('/benchmark/results')
FIGURES_DIR = Path('/benchmark/figures/main')
TMP_DIR = Path('/tmp/fig3_work')

# ── Toggle: True = load pre-computed JSON results; False = run live tests ──

MATRIX_FILE = Path('/benchmark/config/benchmark_testcases.json')
RESULTS_FILE = RESULTS_DIR / 'robustness_benchmark.json'

for d in [RESULTS_DIR, FIGURES_DIR, TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

COLORS = {
    'CrossCell': '#E91E63', 'Zellkonverter': '#2196F3',
    'anndataR': '#4CAF50', 'convert2anndata': '#FF9800',
    'easySCF': '#9C27B0',
    'success': '#4CAF50', 'failed': '#F44336', 'na': '#BDBDBD',
    'mode1_ffi': '#D32F2F', 'mode2_api': '#FF9800', 'mode3_format': '#FFC107',
}

ALL_TOOLS = ['CrossCell', 'Zellkonverter', 'anndataR', 'convert2anndata', 'easySCF']

with open(MATRIX_FILE) as f:
    test_matrix = json.load(f)

print('✅ Environment setup complete')


✅ Environment setup complete


## 1. Helper Functions

In [2]:
# -- Helper: run command with /usr/bin/time --
def run_timed(cmd, timeout=600):
    time_cmd = ['/usr/bin/time', '-v'] + (cmd if isinstance(cmd, list) else cmd.split())
    start = time.time()
    try:
        r = subprocess.run(time_cmd, capture_output=True, text=True, timeout=timeout)
        elapsed = time.time() - start
    except subprocess.TimeoutExpired:
        return False, timeout, 0, 'TIMEOUT'
    peak = 0
    m = re.search(r'Maximum resident set size \(kbytes\): (\d+)', r.stderr)
    if m:
        peak = int(m.group(1)) / 1024.0
    return r.returncode == 0, elapsed, peak, r.stderr

def extract_error_reason(stderr_text):
    """Extract a short error reason from R/tool stderr output."""
    if not stderr_text:
        return 'unknown error'
    for line in stderr_text.split('\n'):
        line = line.strip()
        if line.startswith('Error') or 'error' in line.lower():
            return line[:120] + ('...' if len(line) > 120 else '')
    lines = [l.strip() for l in stderr_text.strip().split('\n') if l.strip()]
    if lines:
        return lines[-1][:120]
    return 'unknown error'

# -- Tool runners --
def run_crosscell(input_path, output_path, fmt):
    ok, t, mem, stderr = run_timed(['crosscell', 'convert', '-i', input_path, '-o', output_path, '-f', fmt])
    return ok, t, mem, stderr

def run_r_tool(input_path, output_path, tool_name):
    scripts = {
        'Zellkonverter': (
            f'suppressPackageStartupMessages({{library(zellkonverter);library(Seurat);library(SingleCellExperiment)}})\n'
            f'obj <- readRDS("{input_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'sce <- as.SingleCellExperiment(obj)\nwriteH5AD(sce, "{output_path}")'
        ),
        'anndataR': (
            f'suppressPackageStartupMessages({{library(anndataR);library(Seurat)}})\n'
            f'obj <- readRDS("{input_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'if (file.exists("{output_path}")) file.remove("{output_path}")\n'
            f'ad <- as_AnnData(obj)\nwrite_h5ad(ad, "{output_path}")'
        ),
        'convert2anndata': (
            f'suppressPackageStartupMessages({{library(convert2anndata);library(Seurat)}})\n'
            f'obj <- readRDS("{input_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'sce <- convert_seurat_to_sce(obj)\nad <- convert_to_anndata(sce)\n'
            f'anndata::write_h5ad(ad, "{output_path}")'
        ),
        'easySCF': (
            f'suppressPackageStartupMessages({{library(easySCFr);library(Seurat)}})\n'
            f'obj <- readRDS("{input_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'saveH5(obj, "{output_path}")'
        ),
    }
    if tool_name not in scripts:
        return False, 0, 0, 'unsupported tool'
    ok, t, mem, stderr = run_timed(['Rscript', '-e', scripts[tool_name]])
    return ok, t, mem, stderr

def run_r_tool_h2r(input_path, output_path, tool_name):
    """H5AD -> RDS via R tools (Zellkonverter/anndataR/easySCF)."""
    if tool_name == 'easySCF':
        h5_tmp = output_path.replace('.rds', '_easyscf.h5')
        py_script = (
            f'import scanpy as sc; from easySCFpy import saveH5; '
            f'adata = sc.read_h5ad("{input_path}"); '
            f'saveH5(adata, "{h5_tmp}")'
        )
        r_script = (
            f'suppressPackageStartupMessages({{library(easySCFr);library(Seurat)}})\n'
            f'obj <- loadH5("{h5_tmp}")\n'
            f'saveRDS(obj, "{output_path}")'
        )
        ok1, t1, mem1, stderr1 = run_timed(['python3', '-c', py_script], timeout=300)
        if not ok1:
            return False, t1, mem1, f'easySCF step1 (saveH5) failed: {extract_error_reason(stderr1)}'
        ok2, t2, mem2, stderr2 = run_timed(['Rscript', '-e', r_script], timeout=300)
        total_time = t1 + t2
        peak_mem = max(mem1, mem2)
        if not ok2:
            return False, total_time, peak_mem, f'easySCF step2 (loadH5) failed: {extract_error_reason(stderr2)}'
        return True, total_time, peak_mem, ''

    scripts = {
        'Zellkonverter': (
            f'suppressPackageStartupMessages({{library(zellkonverter);library(Seurat);library(SingleCellExperiment)}})\n'
            f'sce <- readH5AD("{input_path}")\n'
            f'# Use first assay as counts (default as.Seurat fails because H5AD X maps to non-counts assay)\n'
            f'obj <- as.Seurat(sce, counts=assayNames(sce)[1], data=NULL)\n'
            f'saveRDS(obj, "{output_path}")'
        ),
        'anndataR': (
            f'suppressPackageStartupMessages({{library(anndataR);library(Seurat)}})\n'
            f'ad <- read_h5ad("{input_path}")\n'
            f'obj <- ad$to_Seurat()\n'
            f'saveRDS(obj, "{output_path}")'
        ),
    }
    if tool_name not in scripts:
        return False, 0, 0, 'unsupported direction'
    ok, t, mem, stderr = run_timed(['Rscript', '-e', scripts[tool_name]])
    return ok, t, mem, stderr

def run_tool_benchmark(tool_key, tool_name, directions):
    """Run benchmark for a single tool. Returns dict with results per direction."""
    results = {}
    with open(MATRIX_FILE) as f:
        testcases = json.load(f)

    for direction in directions:
        cases = testcases['test_cases'].get(direction, [])
        dir_results = []
        print(f'  --- {direction} ({len(cases)} tests) ---')

        for tc in cases:
            test_id = tc['test_id']
            filename = tc['file']
            input_path = str(DATA_DIR / filename)

            if not Path(input_path).exists():
                print(f'  skip {test_id}: file not found')
                dir_results.append({**tc, 'status': 'skipped', 'error': 'file_not_found',
                                    'conversion_time_seconds': 0, 'peak_memory_mb': 0})
                continue

            if direction == 'rds_to_h5ad':
                output_path = str(TMP_DIR / f'{tool_key}_{test_id}.h5ad')
                if tool_key == 'crosscell':
                    ok, t, mem, stderr = run_crosscell(input_path, output_path, 'anndata')
                else:
                    ok, t, mem, stderr = run_r_tool(input_path, output_path, tool_name)
            else:
                output_path = str(TMP_DIR / f'{tool_key}_{test_id}.rds')
                if tool_key == 'crosscell':
                    ok, t, mem, stderr = run_crosscell(input_path, output_path, 'seurat')
                else:
                    ok, t, mem, stderr = run_r_tool_h2r(input_path, output_path, tool_name)

            status = 'success' if ok else 'failed'
            icon = 'OK' if ok else 'FAIL'
            err_msg = ''
            if not ok:
                err_msg = extract_error_reason(stderr)
                print(f'  {icon} {test_id:35s} {t:7.2f}s  {mem:7.0f}MB  {status}  ! {err_msg}')
            else:
                print(f'  {icon} {test_id:35s} {t:7.2f}s  {mem:7.0f}MB  {status}')

            record = {**tc, 'status': status,
                      'conversion_time_seconds': round(t, 3),
                      'peak_memory_mb': round(mem, 1)}
            if err_msg:
                record['error'] = err_msg
            dir_results.append(record)

        results[direction] = dir_results
    return results

print('Helper functions defined')


Helper functions defined


## 2.1 Run CrossCell (rds_to_h5ad + h5ad_to_rds)

In [3]:
print('Running CrossCell benchmark...')
crosscell_results = run_tool_benchmark('crosscell', 'CrossCell', ['rds_to_h5ad', 'h5ad_to_rds'])
print(f'\nCrossCell complete')


Running CrossCell benchmark...
  --- rds_to_h5ad (42 tests) ---
  OK v4_pbmc3k_raw                          0.50s      155MB  success
  OK v4_pbmc3k_processed                    0.61s      233MB  success
  OK v5_pbmc3k_raw                          0.28s      154MB  success
  OK v5_pbmc3k_processed                    0.85s      300MB  success
  OK v4_stxKidney_raw                       1.74s      566MB  success
  OK v5_stxKidney_raw                       1.15s      532MB  success
  OK v4_celegans.embryo_raw                 0.95s      318MB  success
  OK v4_celegans.embryo_processed           1.40s      480MB  success
  OK v5_celegans.embryo_raw                 0.72s      334MB  success
  OK v5_celegans.embryo_processed           1.55s      635MB  success
  OK v4_cbmc_raw                            1.54s      581MB  success
  OK v4_cbmc_processed                      2.85s      718MB  success
  OK v5_cbmc_raw                            1.58s      546MB  success
  OK v5_cbmc_processed    

## 2.2 Run Zellkonverter (rds_to_h5ad + h5ad_to_rds)

In [4]:
print('Running Zellkonverter benchmark...')
zellkonverter_results = run_tool_benchmark('zellkonverter', 'Zellkonverter', ['rds_to_h5ad', 'h5ad_to_rds'])
print(f'\nZellkonverter complete')


Running Zellkonverter benchmark...
  --- rds_to_h5ad (42 tests) ---
  OK v4_pbmc3k_raw                         12.54s     1121MB  success
  OK v4_pbmc3k_processed                   12.46s     1206MB  success
  OK v5_pbmc3k_raw                         10.26s     1070MB  success
  OK v5_pbmc3k_processed                   14.13s     1224MB  success
  OK v4_stxKidney_raw                      14.04s     1558MB  success
  OK v5_stxKidney_raw                      13.79s     1403MB  success
  OK v4_celegans.embryo_raw                12.77s     1321MB  success
  OK v4_celegans.embryo_processed          15.14s     1450MB  success
  OK v5_celegans.embryo_raw                13.20s     1202MB  success
  OK v5_celegans.embryo_processed          14.83s     1578MB  success
  OK v4_cbmc_raw                           15.47s     1595MB  success
  OK v4_cbmc_processed                     16.81s     1788MB  success
  OK v5_cbmc_raw                           13.93s     1414MB  success
  OK v5_cbmc_processed

## 2.3 Run anndataR (rds_to_h5ad + h5ad_to_rds)

In [5]:
print('Running anndataR benchmark...')
anndataR_results = run_tool_benchmark('anndataR', 'anndataR', ['rds_to_h5ad', 'h5ad_to_rds'])
print(f'\nanndataR complete')


Running anndataR benchmark...
  --- rds_to_h5ad (42 tests) ---
  OK v4_pbmc3k_raw                          5.76s      646MB  success
  OK v4_pbmc3k_processed                    6.68s      712MB  success
  OK v5_pbmc3k_raw                          4.81s      585MB  success
  OK v5_pbmc3k_processed                    7.26s      706MB  success
  OK v4_stxKidney_raw                       8.15s     1127MB  success
  OK v5_stxKidney_raw                       6.45s      939MB  success
  OK v4_celegans.embryo_raw                 7.29s      861MB  success
  OK v4_celegans.embryo_processed           9.21s     1027MB  success
  OK v5_celegans.embryo_raw                 6.04s      792MB  success
  OK v5_celegans.embryo_processed           8.56s     1027MB  success
  OK v4_cbmc_raw                            8.27s     1133MB  success
  OK v4_cbmc_processed                     11.92s     1399MB  success
  OK v5_cbmc_raw                            6.89s      985MB  success
  OK v5_cbmc_processed     

## 2.4 Run convert2anndata (rds_to_h5ad)

In [6]:
print('Running convert2anndata benchmark...')
convert2anndata_results = run_tool_benchmark('convert2anndata', 'convert2anndata', ['rds_to_h5ad'])
print(f'\nconvert2anndata complete')


Running convert2anndata benchmark...
  --- rds_to_h5ad (42 tests) ---
  FAIL v4_pbmc3k_raw                          9.78s      760MB  failed  ! Error in value[[3L]](cond) :
  FAIL v4_pbmc3k_processed                    9.85s      886MB  failed  ! Error in value[[3L]](cond) :
  OK v5_pbmc3k_raw                         13.75s      985MB  success
  OK v5_pbmc3k_processed                   14.20s     1532MB  success
  FAIL v4_stxKidney_raw                      10.22s      898MB  failed  ! Error in value[[3L]](cond) :
  OK v5_stxKidney_raw                      14.84s     1426MB  success
  FAIL v4_celegans.embryo_raw                 9.34s      821MB  failed  ! Error in value[[3L]](cond) :
  FAIL v4_celegans.embryo_processed          11.02s     1114MB  failed  ! Error in value[[3L]](cond) :
  OK v5_celegans.embryo_raw                15.15s     1197MB  success
  OK v5_celegans.embryo_processed          19.03s     2176MB  success
  FAIL v4_cbmc_raw                           12.33s      910MB  f

## 2.5 Run easySCF (rds_to_h5ad + h5ad_to_rds)

In [7]:
print('Running easySCF benchmark...')
easySCF_results = run_tool_benchmark('easySCF', 'easySCF', ['rds_to_h5ad', 'h5ad_to_rds'])
print(f'\neasySCF complete')


Running easySCF benchmark...
  --- rds_to_h5ad (42 tests) ---
  OK v4_pbmc3k_raw                          8.23s      665MB  success
  OK v4_pbmc3k_processed                   12.78s      889MB  success
  OK v5_pbmc3k_raw                          7.37s      614MB  success
  OK v5_pbmc3k_processed                    9.47s      813MB  success
  OK v4_stxKidney_raw                       6.74s      650MB  success
  OK v5_stxKidney_raw                       5.65s      698MB  success
  OK v4_celegans.embryo_raw                 9.48s      931MB  success
  OK v4_celegans.embryo_processed          13.39s     1366MB  success
  OK v5_celegans.embryo_raw                 8.65s      811MB  success
  OK v5_celegans.embryo_processed          13.30s     1226MB  success
  OK v4_cbmc_raw                           10.91s     1231MB  success
  OK v4_cbmc_processed                     16.64s     1852MB  success
  OK v5_cbmc_raw                            9.39s     1040MB  success
  OK v5_cbmc_processed      

## 3. Save Results

In [8]:
# Merge all tool results
all_results = {}
for tool_key, tool_name, directions in [
    ('crosscell', 'CrossCell', ['rds_to_h5ad', 'h5ad_to_rds']),
    ('zellkonverter', 'Zellkonverter', ['rds_to_h5ad', 'h5ad_to_rds']),
    ('anndataR', 'anndataR', ['rds_to_h5ad', 'h5ad_to_rds']),
    ('convert2anndata', 'convert2anndata', ['rds_to_h5ad']),
    ('easySCF', 'easySCF', ['rds_to_h5ad', 'h5ad_to_rds']),
]:
    var_name = f'{tool_key}_results'
    if var_name in dir():
        all_results[tool_key] = eval(var_name)
    elif var_name in globals():
        all_results[tool_key] = globals()[var_name]
    else:
        print(f'Warning: {tool_name}: no results found (cell not run?)')

with open(RESULTS_FILE, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Results saved to {RESULTS_FILE}')


Results saved to /benchmark/results/robustness_benchmark.json
